In [1]:
!pip install -q \
langgraph \
langchain \
langchain-community \
langchain-google-genai \
langchain-chroma \
langchain-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.

In [64]:
from google.colab import drive

try:
    drive.mount('/content/drive')
    print("Drive mounted Successfully")
except Exception as e:
    print(f"Drive mount failed: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted Successfully


In [66]:
import os
from google.colab import userdata

try:

    os.environ["GOOGLE_API_KEY"] = (
        userdata.get("Gemini_key")
    )

    GROQ_API_KEY = (
        userdata.get("Groq_Key")
    )

    print("API Keys Loaded Successfully")

except Exception as e:

    print(f"API Key Error: {e}")

API Keys Loaded Successfully


In [67]:
try:
    from typing import TypedDict
    from langgraph.graph import (
        StateGraph,
        START,
        END
    )
    from langchain_chroma import Chroma
    from langchain_google_genai import (
        GoogleGenerativeAIEmbeddings
    )
    from groq import Groq

    print("All Imported successfully")

except Exception as e:
    print(f"Import Error: {e}")

All Imported successfully


Connect ChromaDB

In [68]:
try:

    embeddings = (
        GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001"
        )
    )
    db = Chroma(
        collection_name="wiki_rag",
        embedding_function=embeddings,
        persist_directory=
        "/content/drive/MyDrive/RAG_Internship/chroma_db"
    )
    retriever = db.as_retriever(
        search_kwargs={"k": 5}
    )
    print(
        "Documents:",
        db._collection.count()
    )

except Exception as e:
    print(f"ChromaDB Error: {e}")

Documents: 245


In [69]:
try:
    client = Groq(
        api_key=GROQ_API_KEY
    )
    print("Groq initialized Successfully")

except Exception as e:
    print(f"Groq Error: {e}")

Groq initialized Successfully


In [52]:
class GraphState(TypedDict):

    question: str
    docs: list
    context: str
    answer: str
    error: str

Retrieval Node

In [53]:
def retrieve_node(state):
    try:
        question = state["question"]
        docs = retriever.invoke(
            question
        )
        return {
            "docs": docs
        }
    except Exception as e:
        return {
            "error": str(e)
        }

Validation Node

In [71]:
def validate_node(state):

    try:
        docs = state.get(
            "docs",
            []
        )
        if len(docs) == 0:

            return {
                "error":
                "No relevant documents found."
            }
        return {}

    except Exception as e:
        return {
            "error":
            f"Validation Error: {e}"
        }

Generation Node

In [72]:
def generate_node(state):

    try:

        docs = state["docs"]
        question = state["question"]
        context = "\n\n".join(
            [
                doc.page_content
                for doc in docs
            ]
        )

        prompt = f"""
Answer ONLY using the provided context.

Context:
{context}

Question:
{question}
"""

        response = (
            client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        answer = (
            response
            .choices[0]
            .message
            .content
        )
        return {
            "context": context,
            "answer": answer
        }

    except Exception as e:
        return {
            "error":
            f"Generation Error: {e}"
        }

Routing Function

In [56]:
def route_after_validation(
    state
):

    if state.get("error"):

        return "error"

    return "generate"

Graph

In [73]:
try:
    builder = StateGraph(
        GraphState
    )
    print("Graph Builder Created")

except Exception as e:
    print(f"Builder Error: {e}")

Graph Builder Created


In [74]:
try:
    builder.add_node(
        "retrieve",
        retrieve_node
    )
    builder.add_node(
        "validate",
        validate_node
    )
    builder.add_node(
        "generate",
        generate_node
    )
    print("Nodes Added Successfully")

except Exception as e:
    print(f"Node Error: {e}")

Nodes Added Successfully


In [75]:
try:

    builder.add_edge(
        START,
        "retrieve"
    )
    builder.add_edge(
        "retrieve",
        "validate"
    )

    print("Edges Added Successfully")

except Exception as e:
    print(f"Edge Error: {e}")

Edges Added Successfully


In [76]:
try:

    builder.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "generate":
            "generate",

            "error":
            END
        }
    )
    print("Conditional Edge Added")

except Exception as e:
    print(
        f"Conditional Edge Error: {e}"
    )

Conditional Edge Added


In [77]:
try:

    builder.add_edge(
        "generate",
        END
    )
    graph = builder.compile()

    print("Graph Compiled Successfully")

except Exception as e:
    print(f"Compile Error: {e}")

Graph Compiled Successfully


In [89]:
question = "What is machine learning?"

result = graph.invoke({"question": question})

print(f"Question: {question}")

output = result.get("answer") or result.get("error")

print(f"\nFinal Output:\n{output}")

Question: What is machine learning?

Final Output:
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [88]:
question = "What is AI?"

result = graph.invoke({"question": question})

print(f"Question: {question}")

output = result.get("answer") or result.get("error")

print(f"\nFinal Output:\n{output}")

Question: What is AI?

Final Output:
According to the context, AI (Artificial Intelligence) can be defined in several ways:

1. The capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.
2. "The computational part of the ability to achieve goals in the world" (McCarthy's definition).
3. "The ability to solve hard problems" (Minsky's definition).
4. The study of agents that perceive their environment and take actions that maximize their chances of achieving defined goals (definition from "Artificial Intelligence: A Modern Approach").
5. An engineered system that generates outputs such as content, forecasts, recommendations, or decisions for a given set of human-defined objectives (International Organization for Standardization definition).
6. A machine-based system that is designed to operate with varying levels of autonomy and that may exhibit adaptiveness after de

In [93]:
question = "Who won FIFA World Cup 2022?"

result = graph.invoke({"question": question})

print(f"Question: {question}")

output = result.get("answer") or result.get("error")

print(f"\nFinal Output:\n{output}")

Question: Who won FIFA World Cup 2022?

Final Output:
The context provided does not mention the FIFA World Cup 2022 or its winner. It primarily discusses advancements in artificial intelligence, machine learning, and their applications in various fields.
